# Hearing Reality — Colab Main Runner

End-to-end notebook for the _Two-Stream Deepfake Audio Detection_ thesis.
Run cells top-to-bottom. Only **Section 1 — Path Configuration** (cell 4) needs manual edits.

| Section | What it does                                               |
| ------- | ---------------------------------------------------------- |
| 1       | Session init: install, Drive mount, path config, GPU check |
| 2       | Fit or load Stream 2 PCA                                   |
| 3       | Train Setup A / B / C (skips if checkpoint exists)         |
| 4       | Extract scores for clean + 6 codec conditions              |
| 5       | Print 3×7 EER% and min-tDCF results tables                 |
| 6       | Measure Real-Time Factor (RTF) per setup                   |


## Section 0 — Dataset Audit (Read-Only)

Verifies that the raw datasets on Google Drive are accessible, readable, and structured correctly.
**Run this before any preprocessing or training. Does not modify any files.**


In [3]:
import os
print(os.listdir('/content/drive/MyDrive'))


['Machine Learning Project', 'CSS125L_Final Project_Laos.ipynb', 'Legado_PA_1_4.ipynb', 'Thesis', '[MJ] PH Traffic', 'SA2_Results', 'Kalbo4Lyf NLP Project', 'FINAL_PA1_6.ipynb', 'OSHS_ChromaDB_Final', 'dataset', '.cache', 'README.md', 'README.txt', 'LICENSE.txt', '.gitattributes', 'To Export ASVSpoof 5 Dataset.ipynb', 'vqa_accessibility_project', '[Thesis] Hearing Reality', 'LJ002-0047_gen.wav', 'LJ002-0055_gen.wav', 'LJ002-0049_gen.wav', 'LJ002-0064_gen.wav', 'LJ002-0052_gen.wav', 'LJ002-0057_gen.wav', 'LJ002-0060_gen.wav', 'LJ002-0058_gen.wav', 'LJ002-0054_gen.wav', 'LJ002-0063_gen.wav', 'LJ002-0059_gen.wav', 'LJ002-0050_gen.wav', 'LJ002-0051_gen.wav', 'LJ002-0062_gen.wav', 'LJ002-0056_gen.wav', 'LJ002-0075_gen.wav', 'LJ002-0071_gen.wav', 'LJ002-0080_gen.wav', 'LJ002-0067_gen.wav', 'LJ002-0076_gen.wav', 'LJ002-0081_gen.wav', 'LJ002-0101_gen.wav', 'LJ002-0099_gen.wav', 'LJ002-0085_gen.wav', 'LJ002-0084_gen.wav', 'LJ002-0090_gen.wav', 'LJ002-0092_gen.wav', 'LJ002-0086_gen.wav', 'LJ002-

In [1]:
# ── Step 1 — Mount Drive and confirm access ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT    = "/content/drive/MyDrive/[Thesis] Hearing Reality"
WAVEFAKE_ROOT = "/content/drive/MyDrive/[Thesis] Hearing Reality/Datasets/Raw/Wavefake"
ASVSPOOF_ROOT = "/content/drive/MyDrive/[Thesis] Hearing Reality/Datasets/Raw/ASV5"

if os.path.exists(DRIVE_ROOT):
    print(f"✅ Drive root found: {DRIVE_ROOT}")
    print(f"   Contents: {os.listdir(DRIVE_ROOT)}")
else:
    print(f"❌ Drive root NOT FOUND: {DRIVE_ROOT}")
    print("   Check that the folder name matches exactly — it is case-sensitive")
    raise SystemExit("Fix Drive folder before proceeding.")

Mounted at /content/drive
✅ Drive root found: /content/drive/MyDrive/[Thesis] Hearing Reality
   Contents: ['Documents', 'Diagrams', 'Manuscript ', 'Python Notebooks', 'Trackers', 'Datasets', 'pca_model', 'Checkpoints']


In [3]:
# ── Step 2 — Walk and count all raw files ─────────────────────────────────────
import os

def audit_folder(root_path, label):
    if not os.path.exists(root_path):
        print(f"❌ {label}: NOT FOUND at {root_path}")
        return {}

    ext_counts = {}
    subdir_counts = {}
    total = 0

    for dirpath, dirnames, filenames in os.walk(root_path):
        rel = os.path.relpath(dirpath, root_path)
        audio_files = [f for f in filenames if f.endswith(('.wav', '.flac', '.mp3', '.ogg'))]
        if audio_files:
            subdir_counts[rel] = len(audio_files)
            total += len(audio_files)
        for f in filenames:
            ext = os.path.splitext(f)[1].lower()
            ext_counts[ext] = ext_counts.get(ext, 0) + 1

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  Root path : {root_path}")
    print(f"  Total audio files: {total}")
    print(f"  File extensions found: {ext_counts}")
    print(f"\n  Subdirectory breakdown:")
    for subdir, count in sorted(subdir_counts.items()):
        print(f"    {subdir:<45} {count:>6} files")

    return subdir_counts

wavefake_counts = audit_folder(WAVEFAKE_ROOT, "WaveFake Dataset")
asvspoof_counts = audit_folder(ASVSPOOF_ROOT, "ASVspoof 5 Dataset")


  WaveFake Dataset
  Root path : /content/drive/MyDrive/[Thesis] Hearing Reality/Datasets/Raw/Wavefake
  Total audio files: 134285
  File extensions found: {'.wav': 134285}

  Subdirectory breakdown:
    generated_audio/common_voices_prompts_from_conformer_fastspeech2_pwg_ljspeech  16283 files
    generated_audio/common_voices_prompts_from_conformer_fastspeech2_pwg_ljspeech/generated  16283 files
    generated_audio/jsut_multi_band_melgan          5000 files
    generated_audio/jsut_parallel_wavegan           5000 files
    generated_audio/ljspeech_full_band_melgan      13100 files
    generated_audio/ljspeech_hifiGAN               13100 files
    generated_audio/ljspeech_melgan                13100 files
    generated_audio/ljspeech_melgan_large          13100 files
    generated_audio/ljspeech_multi_band_melgan     13116 files
    generated_audio/ljspeech_parallel_wavegan      13100 files
    generated_audio/ljspeech_waveglow              13103 files

  ASVspoof 5 Dataset
  Root pat

In [4]:
# ── Step 3 — Verify WaveFake folder structure ─────────────────────────────────
EXPECTED_WAVEFAKE_SUBDIRS = [
    'melgan',
    'parallel_wavegan',
    'waveglow',
    'fullband_melgan',
    'multiband_melgan',
    'hifigan',
]

EXPECTED_REAL_DIRS = ['real', 'ljspeech', 'LJSpeech', 'original']

print("\n── WaveFake Structure Check ──")

if os.path.exists(WAVEFAKE_ROOT):
    found_dirs = [d.lower() for d in os.listdir(WAVEFAKE_ROOT)]

    for expected in EXPECTED_WAVEFAKE_SUBDIRS:
        if expected in found_dirs:
            print(f"  ✅ {expected}")
        else:
            print(f"  ❌ {expected} — MISSING (check folder name spelling)")

    real_found = any(d in found_dirs for d in [r.lower() for r in EXPECTED_REAL_DIRS])
    if real_found:
        matched = [d for d in os.listdir(WAVEFAKE_ROOT) if d.lower() in [r.lower() for r in EXPECTED_REAL_DIRS]]
        print(f"  ✅ Real/bonafide folder found: {matched}")
    else:
        print(f"  ⚠️  No real/bonafide folder found — expected one of: {EXPECTED_REAL_DIRS}")
        print(f"      Actual folders: {os.listdir(WAVEFAKE_ROOT)}")
        print(f"      ACTION NEEDED: Identify which folder contains the real LJSpeech recordings")
else:
    print(f"  ❌ WAVEFAKE_ROOT not found: {WAVEFAKE_ROOT}")


── WaveFake Structure Check ──
  ❌ melgan — MISSING (check folder name spelling)
  ❌ parallel_wavegan — MISSING (check folder name spelling)
  ❌ waveglow — MISSING (check folder name spelling)
  ❌ fullband_melgan — MISSING (check folder name spelling)
  ❌ multiband_melgan — MISSING (check folder name spelling)
  ❌ hifigan — MISSING (check folder name spelling)
  ⚠️  No real/bonafide folder found — expected one of: ['real', 'ljspeech', 'LJSpeech', 'original']
      Actual folders: ['generated_audio']
      ACTION NEEDED: Identify which folder contains the real LJSpeech recordings


In [5]:
# ── Step 4 — Verify ASVspoof 5 structure ──────────────────────────────────────
print("\n── ASVspoof 5 Structure Check ──")

if os.path.exists(ASVSPOOF_ROOT):
    all_items = os.listdir(ASVSPOOF_ROOT)
    print(f"  Top-level contents: {all_items}")

    for partition in ['train', 'dev', 'eval', 'LA']:
        found = any(partition.lower() in item.lower() for item in all_items)
        print(f"  {'✅' if found else '❌'} {partition} partition: {'found' if found else 'NOT FOUND'}")

    label_extensions = ['.txt', '.tsv', '.csv', '.trl']
    label_files = []
    for r, dirs, files in os.walk(ASVSPOOF_ROOT):
        for f in files:
            if any(f.endswith(ext) for ext in label_extensions):
                label_files.append(os.path.relpath(os.path.join(r, f), ASVSPOOF_ROOT))

    if label_files:
        print(f"\n  ✅ Label/metadata files found:")
        for lf in label_files:
            print(f"      {lf}")
    else:
        print(f"\n  ❌ No label/metadata files found — ASVspoof 5 requires a metadata file to assign labels")
        print(f"      Expected filenames like: ASVspoof5.train.metadata.txt, trial_metadata.txt, or *.trl")
else:
    print(f"  ❌ ASVSPOOF_ROOT not found: {ASVSPOOF_ROOT}")


── ASVspoof 5 Structure Check ──
  Top-level contents: ['Flac_E', 'Flac_D', 'ASVspoof5_protocols', 'Flac_T']
  ❌ train partition: NOT FOUND
  ❌ dev partition: NOT FOUND
  ❌ eval partition: NOT FOUND
  ✅ LA partition: found

  ✅ Label/metadata files found:
      ASVspoof5_protocols/ASVspoof5.train.tsv
      ASVspoof5_protocols/ASVspoof5.dev.track_1.tsv
      ASVspoof5_protocols/ASVspoof5.dev.track_2.enroll.tsv
      ASVspoof5_protocols/ASVspoof5.dev.track_2.trial.tsv
      ASVspoof5_protocols/ASVspoof5.eval.track_1.tsv
      ASVspoof5_protocols/ASVspoof5.eval.track_2.enroll.tsv
      ASVspoof5_protocols/ASVspoof5.eval.track_2.trial.tsv
      ASVspoof5_protocols/ASVspoof5.codec.config.csv


In [6]:
# ── Step 5 — Spot-check 5 random audio files per dataset ──────────────────────
import random
import numpy as np
import soundfile as sf

def spot_check_files(root_path, dataset_label, n=5):
    print(f"\n── Spot Check: {dataset_label} ({n} random files) ──")

    all_audio = []
    for r, dirs, files in os.walk(root_path):
        for f in files:
            if f.endswith(('.wav', '.flac', '.mp3')):
                all_audio.append(os.path.join(r, f))

    if not all_audio:
        print(f"  ❌ No audio files found to spot-check")
        return

    sample = random.sample(all_audio, min(n, len(all_audio)))

    for fpath in sample:
        rel = os.path.relpath(fpath, root_path)
        try:
            data, sr = sf.read(fpath)
            duration = len(data) / sr
            rms = float(np.sqrt(np.mean(data**2)))
            has_nan = bool(np.isnan(data).any())
            too_short = duration < 1.0
            silent = rms < 0.001

            issues = []
            if has_nan:    issues.append("NaN values")
            if too_short:  issues.append(f"too short ({duration:.2f}s)")
            if silent:     issues.append("silent (RMS too low)")

            status = "✅" if not issues else "⚠️ "
            print(f"  {status} {rel}")
            print(f"      SR: {sr} Hz | Duration: {duration:.2f}s | RMS: {rms:.4f} | NaN: {has_nan}")
            if issues:
                print(f"      ISSUES: {', '.join(issues)}")

        except Exception as e:
            print(f"  ❌ {rel}")
            print(f"      FAILED TO READ: {e}")

spot_check_files(WAVEFAKE_ROOT, "WaveFake")
spot_check_files(ASVSPOOF_ROOT, "ASVspoof 5")


── Spot Check: WaveFake (5 random files) ──
  ✅ generated_audio/ljspeech_multi_band_melgan/LJ005-0300_gen.wav
      SR: 22050 Hz | Duration: 4.10s | RMS: 0.0582 | NaN: False
  ✅ generated_audio/ljspeech_parallel_wavegan/LJ019-0157_gen.wav
      SR: 22050 Hz | Duration: 5.69s | RMS: 0.0527 | NaN: False
  ✅ generated_audio/common_voices_prompts_from_conformer_fastspeech2_pwg_ljspeech/generated/gen_6522.wav
      SR: 22050 Hz | Duration: 4.28s | RMS: 0.0618 | NaN: False
  ✅ generated_audio/ljspeech_multi_band_melgan/LJ043-0064_gen.wav
      SR: 22050 Hz | Duration: 2.10s | RMS: 0.0495 | NaN: False
  ✅ generated_audio/ljspeech_hifiGAN/LJ007-0142_generated.wav
      SR: 22050 Hz | Duration: 4.56s | RMS: 0.0627 | NaN: False

── Spot Check: ASVspoof 5 (5 random files) ──
  ✅ Flac_T/flac_T Portion 1 (15,000 Files)/T_0000009259.flac
      SR: 16000 Hz | Duration: 16.10s | RMS: 0.0321 | NaN: False
  ✅ Flac_E/flac_E_eval Portion 1 (14,000 Files)/E_0000093347.flac
      SR: 16000 Hz | Duration: 5

In [7]:
# ── Step 6 — Sample rate audit (200 random files per dataset) ─────────────────
import random
import soundfile as sf
from collections import Counter

def sample_rate_audit(root_path, dataset_label, sample_size=200):
    print(f"\n── Sample Rate Audit: {dataset_label} ({sample_size} random files) ──")

    all_audio = []
    for r, dirs, files in os.walk(root_path):
        for f in files:
            if f.endswith(('.wav', '.flac')):
                all_audio.append(os.path.join(r, f))

    if not all_audio:
        print(f"  ❌ No .wav or .flac files found")
        return

    sample = random.sample(all_audio, min(sample_size, len(all_audio)))
    sr_counter = Counter()
    failed = 0

    for fpath in sample:
        try:
            info = sf.info(fpath)
            sr_counter[info.samplerate] += 1
        except:
            failed += 1

    print(f"  Sample rates found:")
    for sr, count in sorted(sr_counter.items()):
        pct = count / len(sample) * 100
        needs_resample = "⚠️  NEEDS RESAMPLING to 16000 Hz" if sr != 16000 else "✅ correct"
        print(f"    {sr} Hz — {count} files ({pct:.1f}%)  {needs_resample}")

    if failed:
        print(f"  ⚠️  {failed} files failed to read")

if os.path.exists(WAVEFAKE_ROOT):
    sample_rate_audit(WAVEFAKE_ROOT, "WaveFake")

if os.path.exists(ASVSPOOF_ROOT):
    sample_rate_audit(ASVSPOOF_ROOT, "ASVspoof 5")


── Sample Rate Audit: WaveFake (200 random files) ──
  Sample rates found:
    22050 Hz — 189 files (94.5%)  ⚠️  NEEDS RESAMPLING to 16000 Hz
    24000 Hz — 11 files (5.5%)  ⚠️  NEEDS RESAMPLING to 16000 Hz

── Sample Rate Audit: ASVspoof 5 (200 random files) ──
  Sample rates found:
    16000 Hz — 200 files (100.0%)  ✅ correct


In [8]:
# ── Step 7 — Full audit summary ───────────────────────────────────────────────
print("\n")
print("═" * 55)
print("  DATASET AUDIT SUMMARY")
print("═" * 55)

wf_exists  = os.path.exists(WAVEFAKE_ROOT)
asv_exists = os.path.exists(ASVSPOOF_ROOT)

print(f"  WaveFake on Drive:    {'✅ Found' if wf_exists else '❌ NOT FOUND'}")
print(f"  ASVspoof 5 on Drive:  {'✅ Found' if asv_exists else '❌ NOT FOUND'}")
print(f"\n  Paths checked:")
print(f"    WAVEFAKE_ROOT = {WAVEFAKE_ROOT}")
print(f"    ASVSPOOF_ROOT = {ASVSPOOF_ROOT}")

print(f"\n  READINESS:")

if wf_exists:
    print(f"  ✅ WaveFake — safe to preprocess and train")
else:
    print(f"  ❌ WaveFake — upload dataset to Drive before proceeding")

if asv_exists:
    print(f"  ✅ ASVspoof 5 — safe to preprocess and train")
else:
    print(f"  ⚠️  ASVspoof 5 — not found; can still train on WaveFake only")

print(f"\n  NEXT STEP:")
if wf_exists or asv_exists:
    print(f"  Run scripts/preprocess_datasets.py to resample,")
    print(f"  generate manifest.csv, and spot-check files.")
else:
    print(f"  ❌ No datasets found. Upload at least WaveFake to Drive first.")

print("═" * 55)



═══════════════════════════════════════════════════════
  DATASET AUDIT SUMMARY
═══════════════════════════════════════════════════════
  WaveFake on Drive:    ✅ Found
  ASVspoof 5 on Drive:  ✅ Found

  Paths checked:
    WAVEFAKE_ROOT = /content/drive/MyDrive/[Thesis] Hearing Reality/Datasets/Raw/Wavefake
    ASVSPOOF_ROOT = /content/drive/MyDrive/[Thesis] Hearing Reality/Datasets/Raw/ASV5

  READINESS:
  ✅ WaveFake — safe to preprocess and train
  ✅ ASVspoof 5 — safe to preprocess and train

  NEXT STEP:
  Run scripts/preprocess_datasets.py to resample,
  generate manifest.csv, and spot-check files.
═══════════════════════════════════════════════════════


## Section 1 — Session Initialisation


In [2]:
# Verify account and GPU
import subprocess

# Check GPU
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB" if torch.cuda.is_available() else "")

# Check which Google account Drive is connected to
from google.colab import drive
drive.mount('/content/drive')

import os
# This will show the Drive root — the folder name confirms the account
print("\nDrive root contents:", os.listdir('/content/drive/MyDrive')[:10])

GPU available: True
GPU name: NVIDIA A100-SXM4-80GB
VRAM: 85.094825984 GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Drive root contents: ['Machine Learning Project', 'CSS125L_Final Project_Laos.ipynb', 'Legado_PA_1_4.ipynb', 'Thesis', '[MJ] PH Traffic', 'SA2_Results', 'Kalbo4Lyf NLP Project', 'FINAL_PA1_6.ipynb', 'OSHS_ChromaDB_Final', 'dataset']


In [3]:
# Check available RAM — Pro gives ~25GB, free gives ~12GB
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f"Total RAM: {ram_gb:.1f} GB")

if ram_gb > 20:
    print("✅ Colab Pro confirmed (>20GB RAM)")
elif ram_gb > 12:
    print("⚠️ Possibly Pro but lower memory runtime selected")
else:
    print("❌ Likely free tier (~12GB RAM) — check your account")

# Check GPU tier
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu}")
    if "A100" in gpu or "V100" in gpu or "A40" in gpu:
        print("✅ Pro-tier GPU confirmed")
    elif "T4" in gpu:
        print("⚠️ T4 GPU — this is available on both free and Pro tiers")
    else:
        print(f"GPU model: {gpu}")

Total RAM: 179.4 GB
✅ Colab Pro confirmed (>20GB RAM)
GPU: NVIDIA A100-SXM4-80GB
✅ Pro-tier GPU confirmed


In [4]:
# ── 1.1  Clone / update repo and install dependencies ────────────────────────
import os, subprocess, sys

REPO_URL    = "https://github.com/Remigaraki/TwoStream-Audio-Forensics.git"
REPO_DIR    = "/content/TwoStream-Audio-Forensics"
REPO_BRANCH = "Rafa's-branch"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, "--single-branch",
                    REPO_URL, REPO_DIR], check=True)
    print("✅ Repo cloned.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", REPO_BRANCH], check=True)
    print("✅ Repo updated.")

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "-q"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ Package installed (editable mode).")
else:
    print("⚠️  Editable install failed — repo is on sys.path so imports will still work.")
    print("   Error:", result.stderr.strip()[-300:])

# Confirm the script is present
script = os.path.join(REPO_DIR, "scripts", "preprocess_datasets.py")
print(f"\n{'✅' if os.path.exists(script) else '❌'} preprocess_datasets.py: {script}")
print(f"CWD: {os.getcwd()}")

✅ Repo cloned.
✅ Package installed (editable mode).

✅ preprocess_datasets.py: /content/TwoStream-Audio-Forensics/scripts/preprocess_datasets.py
CWD: /content/TwoStream-Audio-Forensics


In [ ]:
!python /content/TwoStream-Audio-Forensics/scripts/preprocess_datasets.py \
    --wavefake_root  "/content/drive/MyDrive/[Thesis] Hearing Reality/Datasets/Raw/Wavefake" \
    --asvspoof_root  "/content/drive/MyDrive/[Thesis] Hearing Reality/Datasets/Raw/ASV5" \
    --processed_root "/content/drive/MyDrive/[Thesis] Hearing Reality/Datasets/Processed" \
    --manifest_out   "/content/drive/MyDrive/[Thesis] Hearing Reality/Datasets/manifest.csv" \
    --skip_wavefake --skip_asv_copy       



═══════════════════════════════════════════════════════
  STEP 1 — ASVspoof 5 TSV Schema
═══════════════════════════════════════════════════════
  Reading: /content/drive/MyDrive/[Thesis] Hearing Reality/Datasets/Raw/ASV5/ASVspoof5_protocols/ASVspoof5.train.tsv

  Columns : ['speaker_id', 'utterance_id', 'gender', 'c3', 'c4', 'c5', 'codec', 'system_id', 'label', 'c9']

  First 5 rows:
  speaker_id  utterance_id gender c3 c4 c5 codec system_id  label c9
0     T_4850  T_0000000000      F  -  -  -   AC3       A05  spoof  -
1     T_0858  T_0000000001      M  -  -  -   AC3       A07  spoof  -
2     T_4075  T_0000000002      M  -  -  -   AC2       A04  spoof  -
3     T_0938  T_0000000003      M  -  -  -   AC2       A08  spoof  -
4     T_0610  T_0000000004      M  -  -  -   AC2       A05  spoof  -

  ✅ Columns confirmed:
    filename_col : 'utterance_id'  → e.g. 'T_0000000000'
    label_col    : 'label'  → e.g. 'spoof'
    system_col   : 'system_id'  → e.g. 'A05'

  [SKIP] WaveFake resamplin

In [5]:
# ── 1.2  Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.isdir("/content/drive/MyDrive"), "Drive mount failed"
print("Drive mounted successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted successfully.


In [6]:
# ── CONFIGURATION — edit only this cell ──────────────────────────────────────
DRIVE_ROOT    = "/content/drive/MyDrive/[Thesis] Hearing Reality"
MANIFEST      = f"{DRIVE_ROOT}/Datasets/manifest.csv"
ASVSPOOF_ROOT = f"{DRIVE_ROOT}/Datasets/Raw/ASV5"
PROCESSED     = f"{DRIVE_ROOT}/Datasets/Processed"
PCA_PATH      = f"{DRIVE_ROOT}/pca_model/pca.pkl"
CKPT_ROOT     = f"{DRIVE_ROOT}/Checkpoints"
REPO_ROOT     = "/content/TwoStream-Audio-Forensics"
import sys; sys.path.insert(0, f"{REPO_ROOT}/src")

# Training hyper-parameters
EPOCHS     = 50
BATCH_SIZE = 32
LR         = 1e-4
SEED       = 42

# W&B logging
USE_WANDB     = False
WANDB_PROJECT = "twostream-audio-forensics"

# Which experimental setups to run (any subset of ["A", "B", "C"])
SETUPS_TO_RUN = ["A", "B", "C"]

# ─────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path
Path(PCA_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(CKPT_ROOT).mkdir(parents=True, exist_ok=True)
print("✅ Config loaded")
print(f"   Manifest : {MANIFEST}")
print(f"   PCA path : {PCA_PATH}")
print(f"   Ckpt root: {CKPT_ROOT}")

✅ Config loaded
   Manifest : /content/drive/MyDrive/[Thesis] Hearing Reality/Datasets/manifest.csv
   PCA path : /content/drive/MyDrive/[Thesis] Hearing Reality/pca_model/pca.pkl
   Ckpt root: /content/drive/MyDrive/[Thesis] Hearing Reality/Checkpoints


In [7]:
# ── 1.4  GPU check ───────────────────────────────────────────────────────────
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")

Device: cuda
GPU   : NVIDIA A100-SXM4-80GB
VRAM  : 85.1 GB


## Section 2 — Stream 2 PCA Fitting

Fits the bispectrum PCA basis on the training set and saves it to `CHECKPOINT_DIR/pca.pkl`.  
Re-running this cell is a no-op if the file already exists.


In [31]:
import os
os.remove("/content/drive/MyDrive/[Thesis] Hearing Reality/pca_model/pca.pkl")
print("Deleted. Re-run Section 2.")


Deleted. Re-run Section 2.


In [32]:
# ── 2.1  Fit or reuse PCA on real training audio ─────────────────────────────
import os, subprocess, sys
from pathlib import Path

if os.path.exists(PCA_PATH):
    print(f"✅ PCA already exists — skipping refit.")
    print(f"   Delete the file and re-run this cell to force a refit.")
    print(f"   Path: {PCA_PATH}")
else:
    print("Fitting PCA on real training audio (n_samples=2000) …")
    result = subprocess.run(
        [
            sys.executable,
            f"{REPO_ROOT}/scripts/fit_pca.py",
            "--manifest",  MANIFEST,
            "--split",     "train",
            "--n_samples", "2000",
            "--save_path", PCA_PATH,
            "--data_root", ASVSPOOF_ROOT,
        ],
    )
    if result.returncode != 0:
        raise RuntimeError("fit_pca.py failed — see output above")
    print(f"✅ PCA saved to {PCA_PATH}")

Fitting PCA on real training audio (n_samples=2000) …
✅ PCA saved to /content/drive/MyDrive/[Thesis] Hearing Reality/pca_model/pca.pkl


In [34]:
import joblib
pca = joblib.load(PCA_PATH)
ev = pca._pca.explained_variance_ratio_.sum()
print(f"Components : {pca._pca.n_components_}")
print(f"Explained variance: {ev:.4f}  {'✅' if ev >= 0.95 else '⚠️ below 0.95'}")


Components : 128
Explained variance: 0.9998  ✅


## Section 3 — Training

Trains each setup listed in `SETUPS_TO_RUN`.  
If a checkpoint already exists for a setup, training is skipped — delete the `.pt` file manually to retrain.


In [22]:
# ── 3.1  DataLoader verification (run before training) ───────────────────────
import sys, time, torch
sys.path.insert(0, f"{REPO_ROOT}/src")
from pipeline.dataset import HearingRealityDataset

train_ds = HearingRealityDataset(
    manifest_path=MANIFEST,
    split='train',
    augment=True,
    augment_prob=0.6,
    target_sr=16000,
    clip_duration=4.0,
)
val_ds = HearingRealityDataset(
    manifest_path=MANIFEST,
    split='val',
    augment=False,
    target_sr=16000,
    clip_duration=4.0,
)
print(f"Train size : {len(train_ds)}")
print(f"Val size   : {len(val_ds)}")

loader = torch.utils.data.DataLoader(train_ds, batch_size=32, num_workers=2, shuffle=True)

start = time.time()
for i, (waveforms, labels) in enumerate(loader):
    assert waveforms.shape == (32, 1, 64000), f"Bad shape: {waveforms.shape}"
    assert labels.shape == (32,)
    assert not torch.isnan(waveforms).any(), "NaN in batch"
    if i == 9: break
elapsed = time.time() - start

print(f"10 batches in {elapsed:.1f}s → {10/elapsed:.1f} batches/sec")
print(f"Waveform shape : {waveforms.shape}")
print(f"Label sample   : {labels[:8].tolist()}")
print(f"Label mean     : {labels.float().mean():.3f}  (expected ~0.85–0.90 for 4.2:1 imbalance)")

Train size : 15000
Val size   : 31404
10 batches in 59.9s → 0.2 batches/sec
Waveform shape : torch.Size([32, 1, 64000])
Label sample   : [1, 1, 1, 1, 0, 1, 1, 1]
Label mean     : 0.844  (expected ~0.85–0.90 for 4.2:1 imbalance)


In [21]:
import os, shutil, time
from concurrent.futures import ThreadPoolExecutor, as_completed

src_dir = f"{DRIVE_ROOT}/Datasets/Raw/ASV5/Flac_T/flac_T Portion 1 (15,000 Files)"
dst_dir = "/content/flac_train"
os.makedirs(dst_dir, exist_ok=True)

files = [os.path.join(src_dir, f) for f in os.listdir(src_dir) if f.endswith(".flac")]
print(f"Copying {len(files)} files...")
t0 = time.time()

def _copy(f): shutil.copy2(f, dst_dir)

with ThreadPoolExecutor(max_workers=16) as pool:
    futures = [pool.submit(_copy, f) for f in files]
    done = 0
    for fut in as_completed(futures):
        fut.result()
        done += 1
        if done % 3000 == 0:
            print(f"  {done}/{len(files)} copied...")

elapsed = time.time() - t0
print(f"✅ Done: {len(files)} files in {elapsed:.0f}s ({len(files)/elapsed:.0f} files/sec)")


Copying 15000 files...
  3000/15000 copied...
  6000/15000 copied...
  9000/15000 copied...
  12000/15000 copied...
  15000/15000 copied...
✅ Done: 15000 files in 621s (24 files/sec)


In [20]:
# ── 3.1a  Stage train + val files to local SSD (run once per session) ─────────
# Uses 8 parallel copy workers — Drive FUSE handles concurrent reads much better
# than sequential ones. Typical time: ~3 min for train (15k), ~6 min for val (31k).
import os, shutil, glob, time
from concurrent.futures import ThreadPoolExecutor, as_completed

def _stage(src_dir, dst_dir, label, expected):
    os.makedirs(dst_dir, exist_ok=True)
    existing = len(os.listdir(dst_dir))
    if existing >= expected:
        print(f"✅ {label}: already staged ({existing} files)")
        return
    files = glob.glob(os.path.join(src_dir, "*.flac"))
    print(f"Staging {label}: {len(files)} files  {src_dir} → {dst_dir}")
    t0 = time.time()
    def _copy(f): shutil.copy2(f, dst_dir)
    with ThreadPoolExecutor(max_workers=8) as pool:
        futures = [pool.submit(_copy, f) for f in files]
        done = 0
        for fut in as_completed(futures):
            fut.result()
            done += 1
            if done % 3000 == 0:
                print(f"  {done}/{len(files)} …")
    elapsed = time.time() - t0
    staged = len(os.listdir(dst_dir))
    print(f"✅ {label}: {staged} files in {elapsed:.0f}s ({staged/elapsed:.0f} files/sec)")
    if staged < expected - 100:
        raise RuntimeError(f"⚠️  {label}: expected ~{expected}, got {staged} — check SRC_DIR")

ASV_ROOT = f"{DRIVE_ROOT}/Datasets/Raw/ASV5"

_stage(
    src_dir = f"{ASV_ROOT}/Flac_T/flac_T Portion 1 (15,000 Files)",
    dst_dir = "/content/flac_train",
    label   = "train (T_*.flac)",
    expected = 15000,
)

_stage(
    src_dir = f"{ASV_ROOT}/Flac_D/flac_D Portion 1 (14,200 Files)",
    dst_dir = "/content/flac_val",
    label   = "val Portion 1 (D_*.flac)",
    expected = 14200,
)
_stage(
    src_dir = f"{ASV_ROOT}/Flac_D/flac_D Portion 2 (7,854 Files)",
    dst_dir = "/content/flac_val",
    label   = "val Portion 2 (D_*.flac)",
    expected = 7854,
)
_stage(
    src_dir = f"{ASV_ROOT}/Flac_D/flac_D Portion 2.5 (9,300 Files)",
    dst_dir = "/content/flac_val",
    label   = "val Portion 2.5 (D_*.flac)",
    expected = 9300,
)

Staging train (T_*.flac): 0 files  /content/drive/MyDrive/[Thesis] Hearing Reality/Datasets/Raw/ASV5/Flac_T/flac_T Portion 1 (15,000 Files) → /content/flac_train
✅ train (T_*.flac): 0 files in 0s (0 files/sec)


RuntimeError: ⚠️  train (T_*.flac): expected ~15000, got 0 — check SRC_DIR

In [24]:
# Local SSD roots for train and val (set by staging cell above)
TRAIN_DATA_ROOT = "/content/flac_train"
VAL_DATA_ROOT   = "/content/flac_val"
print(f"TRAIN_DATA_ROOT = {TRAIN_DATA_ROOT}")
print(f"VAL_DATA_ROOT   = {VAL_DATA_ROOT}")

TRAIN_DATA_ROOT = /content/flac_train
VAL_DATA_ROOT   = /content/flac_val


In [31]:
TRAIN_DATA_ROOT = "/content/flac_train"
VAL_DATA_ROOT   = None


In [ ]:
import os, shutil, time
from concurrent.futures import ThreadPoolExecutor, as_completed

src_dir = f"{DRIVE_ROOT}/Datasets/Raw/ASV5/Flac_T/flac_T Portion 1 (15,000 Files)"
dst_dir = "/content/flac_train"
os.makedirs(dst_dir, exist_ok=True)

files = [os.path.join(src_dir, f) for f in os.listdir(src_dir) if f.endswith(".flac")]
print(f"Copying {len(files)} files...")
t0 = time.time()

def _copy(f): shutil.copy2(f, dst_dir)

with ThreadPoolExecutor(max_workers=16) as pool:
    futures = [pool.submit(_copy, f) for f in files]
    done = 0
    for fut in as_completed(futures):
        fut.result()
        done += 1
        if done % 3000 == 0:
            print(f"  {done}/{len(files)} copied...")

elapsed = time.time() - t0
print(f"✅ Done: {len(files)} files in {elapsed:.0f}s ({len(files)/elapsed:.0f} files/sec)")


In [ ]:
# ── 3.2  Launch Setup A training ─────────────────────────────────────────────
# Change SETUP to "B" or "C" for subsequent runs.
import subprocess, sys

SETUP    = "A"
ckpt_dir = f"{CKPT_ROOT}/setup_{SETUP.lower()}"

cmd = [
    sys.executable, f"{REPO_ROOT}/src/train.py",
    "--setup",            SETUP,
    "--epochs",           str(EPOCHS),
    "--lr",               str(LR),
    "--batch_size",       str(BATCH_SIZE),
    "--manifest",         MANIFEST,
    "--pca_path",         PCA_PATH,
    "--ckpt_dir",         ckpt_dir,
    "--train_data_root",  TRAIN_DATA_ROOT,
]
if VAL_DATA_ROOT is not None:
    cmd += ["--val_data_root", VAL_DATA_ROOT]
if USE_WANDB:
    cmd.append("--use_wandb")

print(f"Launching Setup {SETUP} …  ckpt_dir: {ckpt_dir}")
subprocess.run(cmd)

print(f"\n── Resume command (paste below if session disconnects) ──")
resume_cmd = (
    f"!python {REPO_ROOT}/src/train.py"
    f" --setup {SETUP}"
    f' --resume_from "{ckpt_dir}/latest.pt"'
    f" --epochs {EPOCHS} --lr {LR} --batch_size {BATCH_SIZE}"
    f' --manifest "{MANIFEST}"'
    f' --pca_path "{PCA_PATH}"'
    f' --ckpt_dir "{ckpt_dir}"'
    f' --train_data_root "{TRAIN_DATA_ROOT}"'
)
if VAL_DATA_ROOT is not None:
    resume_cmd += f' --val_data_root "{VAL_DATA_ROOT}"'
if USE_WANDB:
    resume_cmd += " --use_wandb"
print(resume_cmd)


## Section 4 — Score Extraction

Runs inference on the **clean** eval split and all **6 codec conditions** for each setup.  
Writes one CSV per (setup, condition) pair to `RESULTS_DIR`.  
Skips any CSV that already exists.


In [ ]:
# ── 4.1  Define eval conditions ──────────────────────────────────────────────
from pathlib import Path

TORTURED_BASE = f"{DRIVE_ROOT}/Datasets/Tortured"
RESULTS_DIR   = f"{CKPT_ROOT}/results"
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

CODEC_CONDITIONS = ["opus_16k", "opus_32k", "opus_64k", "mp3_64k", "mp3_128k", "aac_128k"]

# (condition_label, audio_root, manifest_split)
ALL_EVAL_CONDITIONS = [("clean", ASVSPOOF_ROOT, "test")] + [
    (label, f"{TORTURED_BASE}/{label}", "test")
    for label in CODEC_CONDITIONS
]

print(f"Eval conditions ({len(ALL_EVAL_CONDITIONS)}):")
for label, root, split in ALL_EVAL_CONDITIONS:
    exists = Path(root).exists()
    print(f"  {label:<12}  {'OK' if exists else 'MISSING'}  {root}")

In [ ]:
# ── 4.2  Score extraction loop ───────────────────────────────────────────────
import subprocess, sys
from pathlib import Path

score_csv_map = {}  # {(setup, condition): csv_path}

for setup in SETUPS_TO_RUN:
    ckpt_path = f"{CKPT_ROOT}/setup_{setup.lower()}/best.pt"
    if not Path(ckpt_path).exists():
        print(f"[Setup {setup}] No checkpoint at {ckpt_path} — skipping.")
        continue

    for condition_label, eval_root, split in ALL_EVAL_CONDITIONS:
        out_csv = f"{RESULTS_DIR}/scores_{setup}_{condition_label}.csv"
        score_csv_map[(setup, condition_label)] = out_csv

        if Path(out_csv).exists():
            print(f"[{setup}/{condition_label}] CSV exists — skipping.")
            continue
        if not Path(eval_root).exists():
            print(f"[{setup}/{condition_label}] Audio dir missing — skipping.")
            continue

        print(f"[{setup}/{condition_label}] Extracting scores …")
        subprocess.run([
            sys.executable, f"{REPO_ROOT}/src/eval/extract_scores.py",
            "--setup",      setup,
            "--ckpt_path",  ckpt_path,
            "--data_root",  eval_root,
            "--manifest",   MANIFEST,
            "--split",      split,
            "--pca_path",   PCA_PATH,
            "--output_csv", out_csv,
            "--batch_size", str(BATCH_SIZE),
        ])
        print(f"  -> {out_csv}")

print("\nScore extraction complete.")

## Section 5 — Evaluation Results Table

Loads all score CSVs and prints a **3 × 7** results table (3 setups × 7 conditions).  
Metrics: EER% (lower is better) and min-tDCF (lower is better).


In [ ]:
import csv
import numpy as np
from src.utils.metrics import compute_eer, compute_t_dcf

CONDITION_ORDER = ["clean"] + CODEC_CONDITIONS

results = {}  # {(setup, condition): {"eer": float, "tdcf": float}}

for setup in SETUPS_TO_RUN:
    for cond in CONDITION_ORDER:
        csv_path = score_csv_map.get((setup, cond))
        if csv_path is None or not Path(csv_path).exists():
            results[(setup, cond)] = {"eer": float("nan"), "tdcf": float("nan")}
            continue

        y_true, y_score = [], []
        with open(csv_path, newline="", encoding="utf-8") as fh:
            for row in csv.DictReader(fh):
                y_true.append(int(row["label"]))
                y_score.append(float(row["score"]))

        y_true  = np.array(y_true)
        y_score = np.array(y_score)

        try:
            eer,  _ = compute_eer(y_true, y_score)
            tdcf, _ = compute_t_dcf(y_true, y_score)
        except Exception as exc:
            print(f"  WARNING [{setup}/{cond}]: {exc}")
            eer, tdcf = float("nan"), float("nan")

        results[(setup, cond)] = {"eer": eer, "tdcf": tdcf}

# ── Print tables ─────────────────────────────────────────────────────────────
COL_W = 12
header = f"{'':8}" + "".join(f"{c:>{COL_W}}" for c in CONDITION_ORDER)
sep    = "-" * len(header)

print("\n" + "=" * len(header))
print("EER (%) — lower is better")
print("=" * len(header))
print(header)
print(sep)
for setup in SETUPS_TO_RUN:
    row = f"Setup {setup}  "
    for cond in CONDITION_ORDER:
        v = results.get((setup, cond), {}).get("eer", float("nan"))
        row += f"{v*100:>{COL_W}.2f}" if not np.isnan(v) else f"{'N/A':>{COL_W}}"
    print(row)

print("\n" + "=" * len(header))
print("min-tDCF — lower is better")
print("=" * len(header))
print(header)
print(sep)
for setup in SETUPS_TO_RUN:
    row = f"Setup {setup}  "
    for cond in CONDITION_ORDER:
        v = results.get((setup, cond), {}).get("tdcf", float("nan"))
        row += f"{v:>{COL_W}.4f}" if not np.isnan(v) else f"{'N/A':>{COL_W}}"
    print(row)
print("=" * len(header))

## Section 6 — Real-Time Factor (RTF) Measurement

Times a single forward pass for each setup and computes `RTF = inference_time / audio_duration`.  
**Target: RTF < 1.0** for real-time capable deployment.


In [ ]:
# ── 6.1  Real-Time Factor (RTF) measurement ──────────────────────────────────
import time, torch, sys
from pathlib import Path
sys.path.insert(0, f"{REPO_ROOT}/src")
from models.build import build_model

AUDIO_DURATION_S = 4.0
AUDIO_SAMPLES    = 64000
STATS_DIM        = 248       # 120 LFCC + 128 PCA
N_WARMUP         = 5
N_TIMED          = 20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | Audio: {AUDIO_DURATION_S}s | Warmup: {N_WARMUP} | Timed: {N_TIMED}\n")

rtf_results = {}

for setup in SETUPS_TO_RUN:
    ckpt_path = f"{CKPT_ROOT}/setup_{setup.lower()}/best.pt"
    rtf_model = build_model(setup)

    if Path(ckpt_path).exists():
        ckpt = torch.load(ckpt_path, map_location=device)
        rtf_model.load_state_dict(ckpt.get("model_state_dict", ckpt))
        print(f"[Setup {setup}] Loaded checkpoint weights.")
    else:
        print(f"[Setup {setup}] No checkpoint — timing random weights.")

    rtf_model = rtf_model.to(device).eval()
    dummy_audio = torch.randn(1, 1, AUDIO_SAMPLES, device=device)
    dummy_stats = torch.randn(1, STATS_DIM, device=device)

    with torch.no_grad():
        for _ in range(N_WARMUP):
            rtf_model(dummy_audio, dummy_stats)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(N_TIMED):
            rtf_model(dummy_audio, dummy_stats)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

    avg_ms = elapsed / N_TIMED * 1000
    rtf    = (elapsed / N_TIMED) / AUDIO_DURATION_S
    rtf_results[setup] = rtf
    print(f"  Setup {setup}: {avg_ms:.2f} ms  RTF={rtf:.4f}  "
          f"({'real-time capable' if rtf < 1.0 else 'SLOWER THAN REAL-TIME'})")

print("\nRTF Summary:")
for setup, rtf in rtf_results.items():
    print(f"  Setup {setup}: RTF = {rtf:.4f}")